In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high_small/checkpoints/post_cell_20_try_3.pickle

In [4]:
%%RecordEvent
%%time
### cell 21 ###

# cell 21 optimized

# define the question and (possibly blank) x-axis title
question_of_interest_cell33 = 'For how many years have you been writing code and/or programming?'
title_for_x_axis_cell33 = ''

# combine percentages for multiple years into one DataFrame using vectorized GPU operations
def combine_subset_of_data_from_multiple_years_33(question_of_interest, x_axis_title, include_2017=False):
    year_sources = [
        ('2018', responses_df_2018_cell10, 'How long have you been writing code to analyze data?'),
        ('2019', responses_df_2019_cell10, 'How long have you been writing code to analyze data (at work or at school)?'),
        ('2020', responses_df_2020, None),
        ('2021', responses_df_2021, None),
        ('2022', responses_df_2022_cell10, None),
    ]
    if include_2017:
        year_sources.insert(0, ('2017', responses_df_2017, None))

    # concatenate and rename in one step
    dfs = []
    for year, df_src, orig_col in year_sources:
        if orig_col:
            df_tmp = (
                df_src[[orig_col]]
                .rename(columns={orig_col: question_of_interest})
                .assign(year=year)
            )
        else:
            df_tmp = (
                df_src[[question_of_interest]]
                .assign(year=year)
            )
        dfs.append(df_tmp)
    df_combined = pd.concat(dfs, ignore_index=True)

    # year-specific recoding via vectorized replace on GPU
    replace_maps = {
        '2018': {
            '< 1 year': '< 1 years',
            'I have never written code but I want to learn': 'I have never written code',
            'I have never written code and I do not want to learn': 'I have never written code',
            '1-2 years': '1-3 years',
            '20-30 years': '20+ years',
            '30-40 years': '20+ years',
            '40+ years': '20+ years',
        },
        '2019': {'1-2 years': '1-3 years'},
        '2020': {'1-2 years': '1-3 years'},
    }
    for yr, mapping in replace_maps.items():
        mask = df_combined['year'] == yr
        df_combined.loc[mask, question_of_interest] = (
            df_combined.loc[mask, question_of_interest]
                       .replace(mapping)
        )

    # compute counts and percentages with GPU groupby
    df_counts = (
        df_combined
        .groupby(['year', question_of_interest])
        .size()
        .reset_index(name='count')
    )
    df_counts['percentage'] = (
        df_counts['count'] * 100 /
        df_counts.groupby('year')['count'].transform('sum')
    ).round(1)

    # rename category column if needed
    label = x_axis_title or ''
    df_counts = df_counts.rename(columns={question_of_interest: label})
    return df_counts[[label, 'percentage', 'year']]

# run and inspect
programming_df_combined = (
    combine_subset_of_data_from_multiple_years_33(
        question_of_interest_cell33,
        title_for_x_axis_cell33
    )
    .sort_values(['year', 'percentage'])
)
programming_df_combined.info()

<class 'cudf.core.dataframe.DataFrame'>
Index: 35 entries, 2 to 28
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0               35 non-null     object
 1   percentage  35 non-null     float64
 2   year        35 non-null     object
dtypes: float64(1), object(2)
memory usage: 1.4+ KB
CPU times: user 411 ms, sys: 43.8 ms, total: 455 ms
Wall time: 455 ms


In [ ]:
%Checkpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high_small/checkpoints/post_cell_21_try_3.pickle